# AI Message Guard — Complete Jupyter Notebook

This notebook trains and tests an SMS spam/ham classifier using TF-IDF + Multinomial Naive Bayes.

**Run every cell from top to bottom.** Internet is required for the first dataset download.

**Safety limitation:** this is an educational SMS spam classifier, not a dedicated phishing detector. A HAM prediction does not prove a message is safe.

## 1. Install required libraries
Run this cell once. If imports fail afterward, restart the kernel and run the notebook again.

In [ ]:
%pip install -q pandas scikit-learn matplotlib joblib

## 2. Import libraries and create output folders
Expected: confirmation messages, no errors.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO
import json

import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

ROOT = Path.cwd() / "ai_message_guard_outputs"
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "model"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

print("Imports successful.")
print("Output folder:", ROOT.resolve())

## 3. Download and load the dataset
Expected: dataset confirmation, row count, and first five messages.

In [ ]:
dataset_path = DATA_DIR / "SMSSpamCollection"
if not dataset_path.exists():
    print("Downloading UCI SMS Spam Collection...")
    with urlopen(DATA_URL, timeout=60) as response:
        zip_data = response.read()
    with ZipFile(BytesIO(zip_data)) as archive:
        member = next((n for n in archive.namelist() if n.endswith("SMSSpamCollection")), None)
        if member is None:
            raise FileNotFoundError("SMSSpamCollection file was not found in the ZIP.")
        dataset_path.write_bytes(archive.read(member))
else:
    print("Using previously downloaded dataset.")

df = pd.read_csv(dataset_path, sep="\t", header=None,
                 names=["label", "message"], encoding="utf-8")
df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str).str.strip()
df = df[df["label"].isin(["ham", "spam"])]
df = df[df["message"] != ""].drop_duplicates().reset_index(drop=True)

print("Dataset loaded. Total messages:", len(df))
display(df.head())

## 4. Inspect class counts
Expected: ham/spam counts and a bar chart. The classes are imbalanced, so accuracy alone is not enough.

In [ ]:
display(df["label"].value_counts().rename_axis("label").to_frame("count"))
df["label"].value_counts().plot(kind="bar", rot=0)
plt.title("Messages per class")
plt.xlabel("Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 5. Split into training and test sets
80% is used for training and 20% for testing. Stratification preserves class proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"],
    test_size=0.20, random_state=42, stratify=df["label"]
)
print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

## 6. Train the model
TF-IDF converts text into numerical features; Naive Bayes learns patterns from the training messages.

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, strip_accents="unicode",
        ngram_range=(1, 2), min_df=2,
        max_features=30000, sublinear_tf=True
    )),
    ("classifier", MultinomialNB(alpha=0.5))
])
model.fit(X_train, y_train)
print("Model training complete!")

## 7. Evaluate on unseen test data
Use these actual results in your portfolio; do not invent or hardcode scores.

In [ ]:
y_pred = model.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "spam_precision": precision_score(y_test, y_pred, pos_label="spam", zero_division=0),
    "spam_recall": recall_score(y_test, y_pred, pos_label="spam", zero_division=0),
    "spam_f1": f1_score(y_test, y_pred, pos_label="spam", zero_division=0)
}
print(f"Accuracy:       {metrics['accuracy']:.3f} ({metrics['accuracy']:.1%})")
print(f"Spam precision: {metrics['spam_precision']:.3f} ({metrics['spam_precision']:.1%})")
print(f"Spam recall:    {metrics['spam_recall']:.3f} ({metrics['spam_recall']:.1%})")
print(f"Spam F1-score:  {metrics['spam_f1']:.3f} ({metrics['spam_f1']:.1%})")
print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=["ham", "spam"], zero_division=0))

## 8. Confusion matrix
Rows are actual labels; columns are predicted labels. Off-diagonal values are errors.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=["ham", "spam"],
    display_labels=["Ham", "Spam"], values_format="d"
)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

## 9. Test sample messages
The predictions shown are generated by your trained model and may not always be correct.

In [ ]:
examples = [
    "Hey, are we meeting at the library at 3 pm today?",
    "Congratulations! You have won a free prize. Claim your reward now!",
    "URGENT! Your account has been selected for a reward. Call now."
]
for message in examples:
    pred = model.predict([message])[0]
    scores = dict(zip(model.classes_, model.predict_proba([message])[0]))
    print("-" * 65)
    print("INPUT:", message)
    print("PREDICTION:", pred.upper())
    print("MODEL SCORES:", {k: f"{float(v):.1%}" for k, v in scores.items()})

## 10. Enter your own message
Run this cell, type a message when prompted, and press Enter. If `input()` does not work in your notebook, use the alternative cell below.

In [ ]:
my_message = input("Paste a message to classify: ").strip()
if not my_message:
    print("No message entered. Run this cell again and enter text.")
else:
    pred = model.predict([my_message])[0]
    scores = dict(zip(model.classes_, model.predict_proba([my_message])[0]))
    print("\nINPUT:", my_message)
    print("PREDICTION:", pred.upper())
    print("MODEL SCORES:", {k: f"{float(v):.1%}" for k, v in scores.items()})
    print("Remember: a HAM result does not prove the message is safe.")

### Alternative if `input()` is unsupported
Edit the text in the next cell and run it.

In [ ]:
my_message = "Your example message goes here"  # Replace this text
pred = model.predict([my_message])[0]
scores = dict(zip(model.classes_, model.predict_proba([my_message])[0]))
print("INPUT:", my_message)
print("PREDICTION:", pred.upper())
print("MODEL SCORES:", {k: f"{float(v):.1%}" for k, v in scores.items()})

## 11. Save the model and evaluation metrics
Expected: two file paths printed below.

In [ ]:
model_path = MODEL_DIR / "spam_classifier.joblib"
metrics_path = MODEL_DIR / "metrics.json"
joblib.dump(model, model_path)
results = {
    "dataset": "UCI SMS Spam Collection",
    "total_messages": int(len(df)),
    "train_messages": int(len(X_train)),
    "test_messages": int(len(X_test)),
    "accuracy": float(metrics["accuracy"]),
    "precision_spam": float(metrics["spam_precision"]),
    "recall_spam": float(metrics["spam_recall"]),
    "f1_spam": float(metrics["spam_f1"]),
    "confusion_matrix_labels": ["ham", "spam"],
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=["ham", "spam"]).tolist(),
    "random_state": 42,
    "test_size": 0.20
}
metrics_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("Saved model:", model_path.resolve())
print("Saved metrics:", metrics_path.resolve())

## Expected scenario summary
- **Dataset cell:** confirmation, number of messages, and a 5-row preview.
- **Training cell:** `Model training complete!`
- **Evaluation cell:** actual accuracy, spam precision, recall, F1-score and classification report.
- **Confusion matrix cell:** a chart showing correct and incorrect predictions.
- **Message input cell:** `SPAM` or `HAM`, plus model scores.
- **Save cell:** locations of the saved model and metrics JSON.

Exact scores are not hardcoded; they are calculated from your own run.

### Common problems
- **Download fails:** check your internet connection and rerun the dataset cell.
- **ModuleNotFoundError:** run the install cell, restart the kernel, then run all cells.
- **`input()` does not work:** edit `my_message` in the alternative cell.
- **Wrong kernel in VS Code:** use Select Kernel and choose the Python environment where libraries were installed.

Dataset source: [UCI Machine Learning Repository — SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection)